# Exp2 evaluation: aggregated vs temporal (sequence) models

Compares models on the same 15,000 test applicants (RQ2.1), then breaks the results down by applicants' history availability and richness (RQ2.2).

**How to use**
1. Train your models on `split == 'train'` and tune on `split == 'val'` (from `clean_data/splits.csv`).
2. Predict a score for **every val and test applicant** with each model.
3. Put them in `MODELS` below and **Run All**.

If `MODELS` is left empty, the notebook runs a demo with two dummy models so you can see the output format.

In [1]:
import sys
from pathlib import Path
import pandas as pd

HERE = next(p for p in [Path.cwd(), Path.cwd() / "outputs" / "exp2"] if (p / "eval_exp2.py").exists())
sys.path.insert(0, str(HERE))
sys.dont_write_bytecode = True
import eval_exp2 as ev

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 1. Your models

In [2]:
# Fill in: model name -> predictions for EVERY val and test applicant, as either
#   - a DataFrame with columns SK_ID_CURR, score   (score = predicted probability of default), or
#   - the path to a CSV file with those two columns.
# The FIRST model is the baseline that the others are compared against.
MODELS = {
    # "Aggregated": aggregated_df,
    # "Sequence": "path/to/sequence_model.csv",
}
N_BOOT = 1000   # bootstrap resamples for the 95% confidence intervals (200 is faster for a quick look)
SAVE = True     # also save the result tables as CSV files in this folder
SEGMENTS = None  # None = default groups (history_segments.parquet), or a CSV path with SK_ID_CURR, segment

In [3]:
DEMO = not MODELS
if DEMO:
    # Two deliberately simple logistic regressions, trained on train only, so you can see what the output looks like.
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    app = ev.attach_split(pd.read_parquet(ev.CLEAN_DIR / "application.parquet",
                                          columns=["SK_ID_CURR", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "DAYS_BIRTH"]))
    for name, cols in [("Demo small", ["EXT_SOURCE_2", "DAYS_BIRTH"]),
                       ("Demo bigger", ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "DAYS_BIRTH"])]:
        m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression())
        m.fit(app.loc[app.split == "train", cols], app.loc[app.split == "train", "TARGET"])
        scored = app[app.split.isin(["val", "test"])]
        MODELS[name] = pd.DataFrame({"SK_ID_CURR": scored.SK_ID_CURR, "score": m.predict_proba(scored[cols])[:, 1]})
    print("MODELS was empty, so this is a DEMO with two dummy models. These numbers are NOT results.")

MODELS was empty, so this is a DEMO with two dummy models. These numbers are NOT results.


## 2. Results

In [4]:
table, comps, preds = ev.compare_prediction_files(list(MODELS.values()), labels=list(MODELS), n_boot=N_BOOT)

### Test-set metrics
One row per model. Each cell shows **value (95% confidence interval)**. The threshold used for precision, recall and
F1 was chosen on the **validation** set (the value that maximises F1), so the test set stays untouched.

| Metric | Role | Meaning |
|---|---|---|
| `pr_auc` | **primary** | precision-recall area; the most informative with only ~8% defaulters (8% = random) |
| `roc_auc` | secondary | how well the model ranks defaulters above non-defaulters (0.5 = random, 1 = perfect); unaffected by the default rate, and the standard credit-scoring metric |
| `precision` / `recall` / `f1` | operating point | at the chosen threshold: share of flagged who default / share of defaulters flagged / balance of both |

Compare models on `pr_auc` and `roc_auc` (the significance tests below). Precision, recall and F1 depend on where each
model's threshold lands, so read them as one example operating point, not as the model comparison. The max-F1 threshold also
treats a missed defaulter and a wrongly flagged applicant as equally costly; a lender would normally set the cut-off
from the cost of each error and its review capacity.

In [5]:
ev.pretty_metrics(table)

,threshold,pr_auc,roc_auc,precision,recall,f1
model,,,,,,
Demo small,0.114,0.142 (0.130-0.156),0.653 (0.637-0.670),0.155 (0.143-0.169),0.374 (0.349-0.401),0.219 (0.204-0.236)
Demo bigger,0.147,0.183 (0.166-0.203),0.710 (0.695-0.725),0.204 (0.186-0.223),0.323 (0.296-0.348),0.250 (0.230-0.270)


### Is each model better than the baseline?
Same test applicants for every model, so the comparison is paired. The p-value comes from the paired bootstrap for
PR-AUC and from the DeLong test for ROC-AUC. Because several models and two metrics are tested at once, the
**verdict uses the Holm-adjusted p-value** (`p_holm` < 0.05), which keeps the chance of any false "significant" result
at 5%. The 95% CI of the difference is not adjusted, so a CI that just excludes 0 can still be "no significant
difference".

In [6]:
ev.pretty_comparisons(comps)

,model,vs_baseline,metric,difference (95% CI),p_value,p_holm,verdict
0,Demo bigger,Demo small,pr_auc,+0.0412 (+0.0295 to +0.0534),0,0,significantly better
1,Demo bigger,Demo small,roc_auc,+0.0566 (+0.0445 to +0.0690),2.8e-20,2.8e-20,significantly better


### Results by history segment (RQ2.2)
Same metrics for each group of applicants, each model at its own validation threshold (the same threshold for
every segment: one lending policy, not one per group).
**Compare segments on ROC-AUC** (first table): it does not depend on the default rate. PR-AUC (second table) does,
so read each segment's PR-AUC against its own `default_rate`, which is the PR-AUC of a random model.
* `HISTORY_SEGMENT`: internal (Home Credit) and/or external (bureau) history, or none.
* `INTERNAL_RICHNESS`: quartile of the amount of internal history (Q1 = least, Q4 = most).

Groups with fewer than 20 defaulters show no metrics (too few to be reliable). To use your own groups, set `SEGMENTS`
to a CSV with columns `SK_ID_CURR, segment`.

In [7]:
seg = ev.load_segments(SEGMENTS)
by_seg = ev.segment_table(preds, seg, ["CUSTOM_SEGMENT"] if SEGMENTS else ["HISTORY_SEGMENT", "INTERNAL_RICHNESS"])
ev.segment_pivot(by_seg, "roc_auc").style.format({"default_rate": "{:.2%}", "n": "{:,.0f}", "defaulters": "{:,.0f}"}, precision=3)

In [8]:
ev.segment_pivot(by_seg, "pr_auc").style.format({"default_rate": "{:.2%}", "n": "{:,.0f}", "defaulters": "{:,.0f}"}, precision=3)

### Is a model better than the baseline inside each segment? (RQ2.2)
Paired difference vs the baseline (first model) inside each segment, with 95% CI. Segments with fewer than 20
defaulters are skipped. Many tests are run here (segments x models x 2 metrics), so the verdict uses `p_holm`,
adjusted across **all** segment tests. Only claim a segment effect if its verdict is significant. This cell is the
slowest (a few minutes with `N_BOOT = 1000`).

In [9]:
seg_comps = ev.segment_comparisons(preds, seg, ["CUSTOM_SEGMENT"] if SEGMENTS else ["HISTORY_SEGMENT", "INTERNAL_RICHNESS"],
                                   n_boot=N_BOOT) if len(preds) > 1 else pd.DataFrame()
ev.pretty_comparisons(seg_comps)

,grouping,segment,model,vs_baseline,metric,difference (95% CI),p_value,p_holm,verdict
0,HISTORY_SEGMENT,external_only,Demo bigger,Demo small,pr_auc,+0.0408 (-0.0207 to +0.0953),0.13,0.4,no significant difference
1,HISTORY_SEGMENT,external_only,Demo bigger,Demo small,roc_auc,+0.1522 (+0.0724 to +0.2358),0.00017,0.0017,significantly better
2,HISTORY_SEGMENT,internal+external,Demo bigger,Demo small,pr_auc,+0.0484 (+0.0350 to +0.0624),0,0,significantly better
3,HISTORY_SEGMENT,internal+external,Demo bigger,Demo small,roc_auc,+0.0641 (+0.0506 to +0.0787),2.7e-19,3.4e-18,significantly better
4,HISTORY_SEGMENT,internal_only,Demo bigger,Demo small,pr_auc,+0.0104 (-0.0061 to +0.0328),0.2,0.4,no significant difference
5,HISTORY_SEGMENT,internal_only,Demo bigger,Demo small,roc_auc,+0.0072 (-0.0078 to +0.0213),0.31,0.4,no significant difference
6,INTERNAL_RICHNESS,Q1,Demo bigger,Demo small,pr_auc,+0.0484 (+0.0215 to +0.0815),0,0,significantly better
7,INTERNAL_RICHNESS,Q1,Demo bigger,Demo small,roc_auc,+0.0410 (+0.0176 to +0.0646),0.00024,0.0021,significantly better
8,INTERNAL_RICHNESS,Q2,Demo bigger,Demo small,pr_auc,+0.0364 (+0.0086 to +0.0646),0.006,0.036,significantly better
9,INTERNAL_RICHNESS,Q2,Demo bigger,Demo small,roc_auc,+0.0312 (+0.0085 to +0.0518),0.0068,0.036,significantly better


## 3. Save

In [10]:
if SAVE and not DEMO:
    table.to_csv(HERE / "exp2_metrics.csv")              # numeric values incl. CI bounds
    comps.to_csv(HERE / "exp2_comparisons.csv", index=False)
    by_seg.to_csv(HERE / "exp2_by_segment.csv", index=False)
    seg_comps.to_csv(HERE / "exp2_segment_comparisons.csv", index=False)
    print("Saved result tables in", HERE)
else:
    print("Not saved (demo mode or SAVE = False).")

Not saved (demo mode or SAVE = False).
